In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import networkx as nx
from scipy.sparse.csgraph import connected_components as sp_connected_components
from scipy import sparse
import json

In [2]:
def read_resources(n=None):
    resources = {}
    packages_dir = Path("data/packages")
    resources_dir = Path("data/resources")

    package_files = list(packages_dir.glob("*.json"))
    if n is not None:
        package_files = package_files[:n]
    for package_file in tqdm(package_files, desc="Reading packages"):
        package_metadata = json.load(package_file.open())
        for resource in package_metadata["result"]["resources"]:
            resource_name = resource["name"]
            resource_id = resource["id"]
            resource_file = resources_dir / f"{resource_id}.csv"
            if not resource_file.exists():
                print(f"Warning: CSV file for {resource_name} not found. Skipping.")
                continue

            resources[resource_name] = pd.read_csv(resource_file)

    return resources


resources = read_resources()

Reading packages:   0%|          | 0/134 [00:00<?, ?it/s]

Reading packages:  12%|█▏        | 16/134 [00:01<00:10, 10.81it/s]

/tmp/ipykernel_40412/824640647.py:19: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  resources[resource_name] = pd.read_csv(resource_file)
Reading packages:  15%|█▍        | 20/134 [00:03<00:22,  5.04it/s]

Reading packages:  19%|█▊        | 25/134 [00:04<00:16,  6.50it/s]

Reading packages:  24%|██▍       | 32/134 [00:04<00:09, 10.50it/s]

...

Reading packages:  63%|██████▎   | 84/134 [00:09<00:03, 14.33it/s]

Reading packages:  69%|██████▊   | 92/134 [00:09<00:02, 17.26it/s]

/tmp/ipykernel_40412/824640647.py:19: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  resources[resource_name] = pd.read_csv(resource_file)
Reading packages:  72%|███████▏  | 97/134 [00:12<00:08,  4.48it/s]

Reading packages:  75%|███████▌  | 101/134 [00:12<00:05,  6.15it/s]

Reading packages: 100%|██████████| 134/134 [00:15<00:00,  8.61it/s]

In [3]:
def include_column(resources, resource_name, column_name):
    data = resources[resource_name][column_name]
    str_average_length = data.dropna().apply(lambda x: len(str(x))).mean()
    if pd.isna(str_average_length):
        return False

    return str_average_length > 8


def make_column_list(resources):
    column_list = []
    for resource_name, df in tqdm(resources.items()):
        for column_name in df.columns:
            if not include_column(resources, resource_name, column_name):
                continue
            column_list.append(f"{resource_name}::{column_name}")
    return column_list


def get_column_values(resources, column_identifier):
    resource_name, column_name = column_identifier.split("::")
    return (
        resources[resource_name][column_name]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )


def make_vocabulary(resources, column_list):
    vocabulary = set()

    for column_identifier in tqdm(column_list, desc="Building vocabulary"):
        values = get_column_values(resources, column_identifier)
        values = pd.Series(values).dropna().astype(str)

        vocabulary.update(values)

    return pd.Series(sorted(vocabulary), name="vocabulary", dtype=str)


def get_column_features(resources, vocabulary, column_identifier):
    values = get_column_values(resources, column_identifier)
    values = pd.Series(values).dropna().astype(str)
    value_counts = values.value_counts()

    return value_counts

In [4]:
column_list = make_column_list(resources)
vocabulary = make_vocabulary(resources, column_list)
vocabulary

Building vocabulary: 100%|██████████| 1281/1281 [00:18<00:00, 70.90it/s] 


0                                                           
1          "a" class office complex located on the southw...
2                                                     "alex"
3                                             "alpine racer"
4                                                    "atala"
                                 ...                        
7226897                ãcole ãlã©mentaire mathieu-da-costa
7226898                     ãcole ãlã©mentaire paul-demers
7226899            ãcole ãlã©mentaire saint-noã«l-chabanel
7226900                     ãcole ã©lã©mentaire paul-demers
7226901                      école élémentaire gabrielle-roy
Name: vocabulary, Length: 7226902, dtype: object

In [5]:
def build_presence_matrix(resources, vocabulary, column_list):
    vocab_index = {value: j for j, value in enumerate(vocabulary)}
    indices, indptr = [], [0]
    for column_identifier in tqdm(column_list, desc="Building presence matrix"):
        values = get_column_features(resources, vocabulary, column_identifier).index
        cols = {vocab_index[v] for v in values if v in vocab_index}
        indices.extend(sorted(cols))
        indptr.append(len(indices))
    indices = np.asarray(indices, dtype=np.int32)
    return sparse.csr_matrix(
        (np.ones(len(indices), dtype=np.int32), indices, np.asarray(indptr)),
        shape=(len(column_list), len(vocabulary)),
    )


def filter_matrix_high(matrix):
    max_df = max(5, int(0.005 * matrix.shape[0]))
    df = matrix.sum(axis=0).A1
    keep_values = (df >= 2) & (df <= max_df)
    return matrix[:, keep_values]


def build_containment_matrix(matrix, filter_high=True):
    sizes = np.diff(matrix.indptr).astype(np.float64)
    if filter_high:
        matrix = filter_matrix_high(matrix)

    intersections = (matrix @ matrix.T).tocsr()
    inv = np.divide(1.0, sizes, out=np.zeros_like(sizes), where=sizes > 0)
    return intersections, sparse.diags(inv) @ intersections, sizes


presence = build_presence_matrix(resources, vocabulary, column_list)
intersections, containment, sizes = build_containment_matrix(presence)

Building presence matrix: 100%|██████████| 1281/1281 [00:25<00:00, 51.10it/s] 


In [6]:
def exact_intersection(presence, i, j):
    a = presence.indices[presence.indptr[i] : presence.indptr[i + 1]]
    b = presence.indices[presence.indptr[j] : presence.indptr[j + 1]]
    return len(np.intersect1d(a, b, assume_unique=True))


def verify(presence, pairs):
    pairs = [(i, j) for i, j in pairs if i < j]  # upper triangle only
    rows = np.fromiter((i for i, _ in pairs), dtype=np.int32, count=len(pairs))
    cols = np.fromiter((j for _, j in pairs), dtype=np.int32, count=len(pairs))
    data = np.fromiter(
        (exact_intersection(presence, i, j) for i, j in pairs),
        dtype=np.float64,
        count=len(pairs),
    )
    n = presence.shape[0]
    return sparse.coo_matrix((data, (rows, cols)), shape=(n, n)).tocsr()


verified = verify(presence, zip(*intersections.nonzero()))

In [7]:
def make_join_table(
    column_list, intersections, true_sizes, threshold=0.5, min_distinct=8
):
    I = sparse.triu(intersections, k=1).tocoo()
    inter = I.data.astype(np.float64)

    c_ij = inter / true_sizes[I.row]
    c_ji = inter / true_sizes[I.col]

    keep = (
        (np.maximum(c_ij, c_ji) > threshold)
        & (true_sizes[I.row] >= min_distinct)
        & (true_sizes[I.col] >= min_distinct)
    )

    rows = []
    for i, j, n, a, b in zip(
        I.row[keep], I.col[keep], inter[keep], c_ij[keep], c_ji[keep]
    ):
        d1, c1 = column_list[i].split("::", 1)
        d2, c2 = column_list[j].split("::", 1)
        if d1 == d2:
            continue
        rows.append(
            {
                "Dataset 1": d1,
                "Column 1": c1,
                "Dataset 2": d2,
                "Column 2": c2,
                "Score": max(a, b),
            }
        )
    return pd.DataFrame(rows).sort_values("Score", ascending=False)


table = make_join_table(column_list, verified, sizes, threshold=0.1)
table.sort_values(by="Score", ascending=False, inplace=True)
table.to_csv("results.csv", index=False)

In [8]:
def get_joinable_columns_for_column(column_identifier, table):
    dataset_name, column_name = column_identifier.split("::", 1)
    joinable_first = table[
        (table["Dataset 1"] == dataset_name) & (table["Column 1"] == column_name)
    ][["Dataset 2", "Column 2", "Score"]]
    joinable_second = table[
        (table["Dataset 2"] == dataset_name) & (table["Column 2"] == column_name)
    ][["Dataset 1", "Column 1", "Score"]]

    joinable = pd.concat(
        [
            joinable_first.rename(
                columns={"Dataset 2": "Dataset", "Column 2": "Column"}
            ),
            joinable_second.rename(
                columns={"Dataset 1": "Dataset", "Column 1": "Column"}
            ),
        ]
    )
    return joinable.sort_values(by="Score", ascending=False).reset_index(drop=True)


from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

console = Console()


def sample_values(resources, column_identifier, n=3, seed=42):
    values = get_column_values(resources, column_identifier)
    if len(values) == 0:
        return []
    return values.sample(min(n, len(values)), random_state=seed).tolist()


def show_joins(resources, column_identifier, table, n_examples=3):
    dataset_name, column_name = column_identifier.split("::", 1)
    joinable = get_joinable_columns_for_column(column_identifier, table)

    console.print(
        Panel(
            "\n".join(
                f"· {v}"
                for v in sample_values(resources, column_identifier, n_examples)
            )
            or "[dim]no values[/dim]",
            title=f"[bold]{column_name}[/bold]  [dim]{dataset_name}[/dim]",
            border_style="cyan",
            expand=False,
        )
    )

    if len(joinable) == 0:
        console.print("[dim]No joinable columns found.[/dim]")
        return

    out = Table(
        box=box.HORIZONTALS,  # light rule between rows (needs a box, not None)
        show_lines=True,
        padding=(0, 2),  # (vertical, horizontal) cell padding
        pad_edge=False,
        header_style="bold",
        border_style="grey30",
    )
    out.add_column("Score", justify="right", style="green", width=6)
    out.add_column("Column", style="bold")
    out.add_column("Dataset", style="dim", max_width=34, overflow="ellipsis")
    out.add_column("Sample values", max_width=52, overflow="fold")

    for dataset, column, score in joinable.itertuples(index=False, name=None):
        values = sample_values(resources, f"{dataset}::{column}", n_examples)
        out.add_row(
            f"{score:.3f}",
            column,
            dataset,
            "[dim]·[/dim] ".join(values) if values else "[dim]—[/dim]",
        )

    console.print(out)


show_joins(resources, "Neighbourhoods - 4326.csv::Neighbourhood", table, n_examples=3)

KeyError: 'Neighbourhoods - 4326.csv'